# 02 – Datenvorbereitung

## Ziel

Dieses Notebook erzeugt einen reproduzierbaren, geschichteten Trainings- und
Holdout-Datensatz für alle nachfolgenden Modellierungs-Notebooks. Es werden
ausschließlich modellunabhängige Verarbeitungsschritte durchgeführt.
Datenabhängige Transformationen wie Skalierung, Resampling und Featureauswahl
erfolgen später innerhalb der jeweiligen Modell-Pipeline, um Data Leakage zu
vermeiden.

In [1]:
from pathlib import Path 

import numpy as np 
import pandas as pd

from sklearn.model_selection import  train_test_split

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
data_path = DATA_DIR /"raw"/"santander_openml.csv"

santander = pd.read_csv(data_path)


santander.shape

(200000, 201)

## 2.1 Definition von Features und Zielvariable

Die Zielvariable wird von den Eingangsmerkmalen getrennt und numerisch als
`0` beziehungsweise `1` codiert. Die 200 anonymisierten Merkmale bilden die
Featurematrix `X`, während `y` die vorherzusagende Zielvariable enthält.

In [4]:
X = santander.drop(columns=["target"])

y = (
    santander["target"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"false" : 0, "true" : 1})
    .astype("int8")
)

In [5]:
print("Form von X:", X.shape)
print("Form von y:", y.shape)
print()
print("Targetverteilung:")
print(y.value_counts())
print("Datentyp von y :", y.dtype)

Form von X: (200000, 200)
Form von y: (200000,)

Targetverteilung:
target
0    179902
1     20098
Name: count, dtype: int64
Datentyp von y : int8


## 2.2 Optimierung der Feature-Datentypen

Die numerischen Features werden von `float64` in `float32` umgewandelt.


In [6]:
memory_before = X.memory_usage(deep=True).sum() / 1024**2

X = X.astype("float32")

memory_after = X.memory_usage(deep=True).sum() / 1024**2

print(f"Speicherbedarf vorher: {memory_before:.2f} MB")
print(f"Speicherbedarf nachher: {memory_after:.2f} MB")
print()
print(X.dtypes.value_counts())

Speicherbedarf vorher: 305.18 MB
Speicherbedarf nachher: 152.59 MB

float32    200
Name: count, dtype: int64


## 2.3 Train-Test-Split

Der Datensatz wird in 80 % Trainingsdaten und 20 % Testdaten aufgeteilt. 

In [7]:
X_train, X_test, y_train , y_test = train_test_split(
    X,
    y,
    test_size= 0.20,
    stratify = y,
    random_state = 42
)

In [8]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train", y_train.shape)
print("y_test:", y_test.shape)

X_train: (160000, 200)
X_test: (40000, 200)
y_train (160000,)
y_test: (40000,)


In [9]:
train_data = X_train.copy()
train_data["target"] = y_train

test_data = X_test.copy()
test_data["target"] = y_test

print("Trainingsdaten:", train_data.shape)
print("Testdaten:", test_data.shape)

Trainingsdaten: (160000, 201)
Testdaten: (40000, 201)


In [10]:
class_distribution = pd.DataFrame({
    "Gesamt_in_Prozent": y.value_counts(normalize=True) * 100,
    "Training_in_Prozent": y_train.value_counts(normalize=True) * 100,
    "Test_in_Prozent": y_test.value_counts(normalize=True) * 100
}).sort_index()

class_distribution.round(3)

,Gesamt_in_Prozent,Training_in_Prozent,Test_in_Prozent
target,,,
0,89.951,89.951,89.95
1,10.049,10.049,10.05


**Ergebnis:** Der geschichtete Split umfasst 160.000 Trainings- und 40.000
Testbeobachtungen. Das ursprüngliche Klassenverhältnis von rund 89,95 % für
`target = 0` und 10,05 % für `target = 1` wurde in beiden Teilmengen nahezu
exakt erhalten.

## 2.4 Speicherung der vorbereiteten Datensätze

Die Trainings- und Testdaten werden unskaliert im Parquet-Format gespeichert.
Dadurch können alle Modell-Notebooks denselben reproduzierbaren Split verwenden.
Modellabhängige Schritte wie Skalierung, Resampling oder Featureauswahl werden
erst innerhalb der jeweiligen Modell-Pipeline durchgeführt.

In [11]:
train_path = PROCESSED_DIR / "train.parquet"
test_path = PROCESSED_DIR / "test.parquet"

train_data.to_parquet(train_path, compression="snappy")
test_data.to_parquet(test_path, compression="snappy")

print("Gespeichert:", train_path)
print("Gespeichert:", test_path)

Gespeichert: /home/lenovo/Dokumente/Uni/Ada_Projekt/data/processed/train.parquet
Gespeichert: /home/lenovo/Dokumente/Uni/Ada_Projekt/data/processed/test.parquet
